In [1]:
"""
Analysis script for comparing NeurIPS checklist results across different models.
Creates visualizations and statistics to compare model performance.
"""

import os
import json
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict
import numpy as np

# Configuration
OUTPUT_BASE_DIR = "neurips_2024_checklist_analysis"
ANALYSIS_OUTPUT_DIR = os.path.join(OUTPUT_BASE_DIR, "analysis_results")

# Create output directory
os.makedirs(ANALYSIS_OUTPUT_DIR, exist_ok=True)

# Set plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

# Checklist questions
CHECKLIST_QUESTIONS = [
    "Claims",
    "Limitations", 
    "Theory Assumptions and Proofs",
    "Experimental Result Reproducibility",
    "Open access to data and code",
    "Experimental Setting/Details",
    "Experiment Statistical Significance",
    "Experiments Compute Resources",
    "Code Of Ethics",
    "Broader Impacts",
    "Safeguards",
    "Licenses for existing assets",
    "New Assets",
    "Crowdsourcing and Research with Human Subjects",
    "Institutional Review Board (IRB) Approvals or Equivalent for Research with Human Subjects"
]

def get_all_model_directories():
    """Get list of all model directories."""
    if not os.path.exists(OUTPUT_BASE_DIR):
        return []
    
    model_dirs = []
    for item in os.listdir(OUTPUT_BASE_DIR):
        item_path = os.path.join(OUTPUT_BASE_DIR, item)
        if os.path.isdir(item_path) and not item.startswith('.') and item != "analysis_results":
            model_dirs.append(item)
    
    return model_dirs

def load_all_results(model_name):
    """
    Load all JSON results for a specific model.
    Returns: list of dicts with parsed data
    """
    model_dir = os.path.join(OUTPUT_BASE_DIR, model_name)
    
    if not os.path.exists(model_dir):
        print(f"Warning: Model directory not found: {model_dir}")
        return []
    
    results = []
    
    for filename in os.listdir(model_dir):
        if filename.endswith('.json'):
            json_path = os.path.join(model_dir, filename)
            
            try:
                with open(json_path, 'r') as f:
                    data = json.load(f)
                results.append(data)
            except Exception as e:
                print(f"Error loading {json_path}: {e}")
    
    return results

def create_comparison_dataframe(models_data):
    """
    Create a pandas DataFrame with all results for analysis.
    
    Returns: DataFrame with columns:
        - paper_hash
        - model
        - question
        - author_answer
        - llm_answer
        - agreement (True/False)
    """
    rows = []
    
    for model_name, results in models_data.items():
        for paper_data in results:
            paper_hash = paper_data.get('paper_hash', 'unknown')
            
            for question, answers in paper_data.get('questions', {}).items():
                author_ans = answers.get('author_answer', 'Not Found')
                llm_ans = answers.get('llm_answer', 'Not Found')
                
                llm_ans = "NA" if llm_ans == "N/A" else llm_ans
                author_ans = "NA" if author_ans == "N/A" else author_ans
                
                # Skip if either is "Not Found"
                if author_ans == 'Not Found' or llm_ans == 'Not Found':
                    print("Not FOUND", paper_hash )
                    continue

                rows.append({
                    'paper_hash': paper_hash,
                    'model': model_name,
                    'question': question,
                    'author_answer': author_ans,
                    'llm_answer': llm_ans,
                    'agreement': author_ans == llm_ans
                })
    
    return pd.DataFrame(rows)

def calculate_statistics(df):
    """Calculate key statistics from the comparison dataframe."""
    
    stats = {}
    
    # Overall statistics by model
    stats['by_model'] = df.groupby('model').agg({
        'agreement': ['count', 'sum', 'mean']
    }).round(3)
    stats['by_model'].columns = ['total_comparisons', 'agreements', 'accuracy']
    
    # Statistics by question
    stats['by_question'] = df.groupby('question').agg({
        'agreement': ['count', 'sum', 'mean']
    }).round(3)
    stats['by_question'].columns = ['total_comparisons', 'agreements', 'accuracy']
    stats['by_question'] = stats['by_question'].sort_values('accuracy', ascending=False)
    
    # Statistics by model and question
    stats['by_model_question'] = df.groupby(['model', 'question'])['agreement'].mean().unstack()
    
    # Confusion matrix: what LLM said when author said X
    stats['confusion'] = {}
    for model in df['model'].unique():
        model_df = df[df['model'] == model]
        stats['confusion'][model] = pd.crosstab(
            model_df['author_answer'], 
            model_df['llm_answer'],
            normalize='index'
        ).round(3)
    
    return stats

def plot_overall_accuracy(df):
    """Plot overall accuracy by model."""
    fig, ax = plt.subplots(figsize=(10, 6))
    
    accuracy_by_model = df.groupby('model')['agreement'].mean().sort_values(ascending=False)
    
    bars = ax.bar(range(len(accuracy_by_model)), accuracy_by_model.values)
    ax.set_xticks(range(len(accuracy_by_model)))
    ax.set_xticklabels(accuracy_by_model.index, rotation=45, ha='right')
    ax.set_ylabel('Accuracy (Agreement Rate)')
    ax.set_xlabel('Model')
    ax.set_title('Overall Model Accuracy in Checklist Evaluation')
    ax.set_ylim([0, 1])
    
    # Add percentage labels on bars
    for i, (bar, val) in enumerate(zip(bars, accuracy_by_model.values)):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.02, 
                f'{val:.1%}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_OUTPUT_DIR, 'overall_accuracy.png'), dpi=300, bbox_inches='tight')
    plt.close()
    
    print("✓ Saved: overall_accuracy.png")

def plot_accuracy_by_question(df):
    """Plot accuracy by question for each model."""
    fig, ax = plt.subplots(figsize=(14, 10))
    
    # Create pivot table
    pivot = df.groupby(['question', 'model'])['agreement'].mean().unstack()
    
    # Plot heatmap
    sns.heatmap(pivot, annot=True, fmt='.2%', cmap='RdYlGn', 
                center=0.5, vmin=0, vmax=1, ax=ax,
                cbar_kws={'label': 'Agreement Rate'})
    
    ax.set_title('Agreement Rate by Question and Model')
    ax.set_xlabel('Model')
    ax.set_ylabel('Question')
    
    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_OUTPUT_DIR, 'accuracy_by_question_heatmap.png'), 
                dpi=300, bbox_inches='tight')
    plt.close()
    
    print("✓ Saved: accuracy_by_question_heatmap.png")

def plot_question_difficulty(df):
    """Plot questions ranked by difficulty (disagreement rate)."""
    fig, ax = plt.subplots(figsize=(12, 8))
    
    question_accuracy = df.groupby('question')['agreement'].agg(['mean', 'count'])
    question_accuracy = question_accuracy.sort_values('mean')
    
    bars = ax.barh(range(len(question_accuracy)), question_accuracy['mean'].values)
    ax.set_yticks(range(len(question_accuracy)))
    ax.set_yticklabels(question_accuracy.index)
    ax.set_xlabel('Agreement Rate')
    ax.set_title('Questions Ranked by Agreement Rate (Easiest to Hardest)')
    ax.set_xlim([0, 1])
    
    # Color bars based on difficulty
    for i, (bar, val) in enumerate(zip(bars, question_accuracy['mean'].values)):
        if val >= 0.8:
            bar.set_color('green')
        elif val >= 0.5:
            bar.set_color('orange')
        else:
            bar.set_color('red')
        
        # Add percentage label
        ax.text(val + 0.02, bar.get_y() + bar.get_height()/2, 
                f'{val:.1%}', va='center')
    
    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_OUTPUT_DIR, 'question_difficulty.png'), 
                dpi=300, bbox_inches='tight')
    plt.close()
    
    print("✓ Saved: question_difficulty.png")

def plot_confusion_matrices(df, models):
    """Plot confusion matrices for each model."""
    n_models = len(models)
    fig, axes = plt.subplots(1, n_models, figsize=(6*n_models, 5))
    
    if n_models == 1:
        axes = [axes]
    
    for ax, model in zip(axes, models):
        model_df = df[df['model'] == model]
        confusion = pd.crosstab(
            model_df['author_answer'], 
            model_df['llm_answer'],
            normalize='index'
        )
        
        sns.heatmap(confusion, annot=True, fmt='.2%', cmap='Blues', ax=ax,
                   vmin=0, vmax=1, cbar_kws={'label': 'Proportion'})
        ax.set_title(f'{model}\nConfusion Matrix')
        ax.set_xlabel('LLM Answer')
        ax.set_ylabel('Author Answer')
    
    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_OUTPUT_DIR, 'confusion_matrices.png'), 
                dpi=300, bbox_inches='tight')
    plt.close()
    
    print("✓ Saved: confusion_matrices.png")

def plot_answer_distribution(df):
    """Plot distribution of answers (Yes/No/NA) by model, with Author as another model."""
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))

    # --- LLM answers by model ---
    llm_dist = df.groupby(['model', 'llm_answer']).size().unstack(fill_value=0)
    llm_dist_pct = llm_dist.div(llm_dist.sum(axis=1), axis=0) * 100

    # --- Author answers (treated as another model) ---
    author_dist = df.groupby('author_answer').size()
    author_dist_pct = (author_dist / author_dist.sum() * 100).to_frame().T
    author_dist_pct.index = ['Authors']  # fake model name

    # Align columns (Yes / No / NA)
    all_cols = llm_dist_pct.columns.union(author_dist_pct.columns)
    llm_dist_pct = llm_dist_pct.reindex(columns=all_cols, fill_value=0)
    author_dist_pct = author_dist_pct.reindex(columns=all_cols, fill_value=0)

    # Combine
    combined = pd.concat([llm_dist_pct, author_dist_pct])

    # Plot
    combined.plot(
        kind='bar',
        stacked=True,
        ax=ax,
        color=['#95a5a6', '#e74c3c', '#2ecc71']
    )

    ax.set_title('Distribution of Answers')
    ax.set_xlabel('Model')
    ax.set_ylabel('Percentage')
    ax.legend(title='Answer', bbox_to_anchor=(1.05, 1))
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

    plt.tight_layout()
    plt.savefig(
        os.path.join(ANALYSIS_OUTPUT_DIR, 'answer_distribution.png'),
        dpi=300,
        bbox_inches='tight'
    )
    plt.close()

    print("✓ Saved: answer_distribution.png")


def save_statistics_report(stats, models_data):
    """Save a text report with key statistics."""
    report_path = os.path.join(ANALYSIS_OUTPUT_DIR, 'statistics_report.txt')
    
    with open(report_path, 'w') as f:
        f.write("="*60 + "\n")
        f.write("NeurIPS CHECKLIST ANALYSIS - STATISTICS REPORT\n")
        f.write("="*60 + "\n\n")
        
        # Overall statistics
        f.write("OVERALL ACCURACY BY MODEL\n")
        f.write("-"*60 + "\n")
        f.write(stats['by_model'].to_string())
        f.write("\n\n")
        
        # Question statistics
        f.write("ACCURACY BY QUESTION (Top 10 Best)\n")
        f.write("-"*60 + "\n")
        f.write(stats['by_question'].head(10).to_string())
        f.write("\n\n")
        
        f.write("ACCURACY BY QUESTION (Top 10 Worst)\n")
        f.write("-"*60 + "\n")
        f.write(stats['by_question'].tail(10).to_string())
        f.write("\n\n")
        
        # Confusion matrices
        for model, confusion in stats['confusion'].items():
            f.write(f"CONFUSION MATRIX - {model}\n")
            f.write("-"*60 + "\n")
            f.write(confusion.to_string())
            f.write("\n\n")
        
        # Dataset info
        f.write("DATASET INFORMATION\n")
        f.write("-"*60 + "\n")
        for model_name, results in models_data.items():
            f.write(f"{model_name}: {len(results)} papers analyzed\n")
    
    print(f"✓ Saved: statistics_report.txt")

def export_to_csv(df):
    """Export the comparison dataframe to CSV."""
    csv_path = os.path.join(ANALYSIS_OUTPUT_DIR, 'full_comparison.csv')
    df.to_csv(csv_path, index=False)
    print(f"✓ Saved: full_comparison.csv")

def analyze_all_models():
    """Main analysis function."""
    print(f"\n{'='*60}")
    print(f"NeurIPS Checklist Analysis")
    print(f"{'='*60}\n")
    
    # Get all model directories
    models = get_all_model_directories()
    
    if not models:
        print("Error: No model directories found!")
        return
    
    print(f"Found {len(models)} model(s): {', '.join(models)}\n")
    
    # Load all results
    print("Loading results...")
    models_data = {}
    for model in models:
        results = load_all_results(model)
        models_data[model] = results
        print(f"  {model}: {len(results)} papers")
    
    print()
    
    # Create comparison dataframe
    print("Creating comparison dataframe...")
    df = create_comparison_dataframe(models_data)
    print(f"  Total comparisons: {len(df):,}")
    print(f"  Unique papers: {df['paper_hash'].nunique():,}")
    print()
    
    # Calculate statistics
    print("Calculating statistics...")
    stats = calculate_statistics(df)
    print()
    
    # Generate visualizations
    print("Generating visualizations...")
    plot_overall_accuracy(df)
    plot_accuracy_by_question(df)
    plot_question_difficulty(df)
    plot_confusion_matrices(df, models)
    plot_answer_distribution(df)
    print()
    
    # Save reports
    print("Saving reports...")
    save_statistics_report(stats, models_data)
    export_to_csv(df)
    print()
    
    # Print summary
    print("="*60)
    print("SUMMARY")
    print("="*60)
    print(f"Models analyzed: {len(models)}")
    print(f"Total papers: {df['paper_hash'].nunique()}")
    print(f"Total comparisons: {len(df):,}")
    print(f"\nBest performing model:")
    best_model = stats['by_model']['accuracy'].idxmax()
    best_accuracy = stats['by_model']['accuracy'].max()
    print(f"  {best_model}: {best_accuracy:.1%} accuracy")
    print(f"\nEasiest question (highest agreement):")
    easiest = stats['by_question']['accuracy'].idxmax()
    easiest_acc = stats['by_question']['accuracy'].max()
    print(f"  {easiest}: {easiest_acc:.1%}")
    print(f"\nHardest question (lowest agreement):")
    hardest = stats['by_question']['accuracy'].idxmin()
    hardest_acc = stats['by_question']['accuracy'].min()
    print(f"  {hardest}: {hardest_acc:.1%}")
    print(f"\nAll results saved to: {ANALYSIS_OUTPUT_DIR}/")
    print("="*60 + "\n")
    
    return df, stats

# Run the analysis
if __name__ == "__main__":
    df, stats = analyze_all_models()

/home/jovyan/.local/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/jovyan/.local/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (

KeyboardInterrupt



In [8]:
"""
Analysis script for comparing NeurIPS checklist results across different models.
Creates visualizations and statistics to compare model performance.
"""

import os
import json
import math
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict
import numpy as np

# Configuration
OUTPUT_BASE_DIR = "neurips_2024_checklist_analysis"
ANALYSIS_OUTPUT_DIR = os.path.join(OUTPUT_BASE_DIR, "analysis_results_spa")

# Create output directory
os.makedirs(ANALYSIS_OUTPUT_DIR, exist_ok=True)

# Set plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

# Checklist questions
CHECKLIST_QUESTIONS = [
    "Claims",
    "Limitations",
    "Theory Assumptions and Proofs",
    "Experimental Result Reproducibility",
    "Open access to data and code",
    "Experimental Setting/Details",
    "Experiment Statistical Significance",
    "Experiments Compute Resources",
    "Code Of Ethics",
    "Broader Impacts",
    "Safeguards",
    "Licenses for existing assets",
    "New Assets",
    "Crowdsourcing and Research with Human Subjects",
    "Institutional Review Board (IRB) Approvals or Equivalent for Research with Human Subjects"
]

TOTAL_QUESTIONS = len(CHECKLIST_QUESTIONS)  # 15


def get_all_model_directories():
    """Get list of all model directories."""
    if not os.path.exists(OUTPUT_BASE_DIR):
        return []

    model_dirs = []
    for item in os.listdir(OUTPUT_BASE_DIR):
        item_path = os.path.join(OUTPUT_BASE_DIR, item)
        if os.path.isdir(item_path) and not item.startswith('.') and not item.startswith('analysis_results'):
            model_dirs.append(item)

    return model_dirs


def load_all_results(model_name):
    """
    Load all JSON results for a specific model.
    Returns: list of dicts with parsed data
    """
    model_dir = os.path.join(OUTPUT_BASE_DIR, model_name)

    if not os.path.exists(model_dir):
        print(f"Warning: Model directory not found: {model_dir}")
        return []

    results = []

    for filename in os.listdir(model_dir):
        if filename.endswith('.json'):
            json_path = os.path.join(model_dir, filename)

            try:
                with open(json_path, 'r') as f:
                    data = json.load(f)
                results.append(data)
            except Exception as e:
                print(f"Error loading {json_path}: {e}")

    return results


def create_comparison_dataframe(models_data):
    """
    Create a pandas DataFrame with all results for analysis.

    Returns: DataFrame with columns:
        - paper_hash
        - model
        - question
        - author_answer
        - llm_answer
        - agreement (True/False)
    """
    rows = []

    for model_name, results in models_data.items():
        for paper_data in results:
            paper_hash = paper_data.get('paper_hash', 'unknown')

            for question, answers in paper_data.get('questions', {}).items():
                author_ans = answers.get('author_answer', 'Not Found')
                llm_ans = answers.get('llm_answer', 'Not Found')

                llm_ans = "NA" if llm_ans == "N/A" else llm_ans
                author_ans = "NA" if author_ans == "N/A" else author_ans

                # Skip if either is "Not Found"
                if author_ans == 'Not Found' or llm_ans == 'Not Found':
                    print("Not FOUND", paper_hash)
                    continue

                rows.append({
                    'paper_hash': paper_hash,
                    'model': model_name,
                    'question': question,
                    'author_answer': author_ans,
                    'llm_answer': llm_ans,
                    'agreement': author_ans == llm_ans
                })

    return pd.DataFrame(rows)


def calculate_statistics(df):
    """Calculate key statistics from the comparison dataframe."""

    stats = {}

    # Overall statistics by model
    stats['by_model'] = df.groupby('model').agg({
        'agreement': ['count', 'sum', 'mean']
    }).round(3)
    stats['by_model'].columns = ['total_comparisons', 'agreements', 'accuracy']

    # Statistics by question
    stats['by_question'] = df.groupby('question').agg({
        'agreement': ['count', 'sum', 'mean']
    }).round(3)
    stats['by_question'].columns = ['total_comparisons', 'agreements', 'accuracy']
    stats['by_question'] = stats['by_question'].sort_values('accuracy', ascending=False)

    # Statistics by model and question
    stats['by_model_question'] = df.groupby(['model', 'question'])['agreement'].mean().unstack()

    # Confusion matrix: what LLM said when author said X
    stats['confusion'] = {}
    for model in df['model'].unique():
        model_df = df[df['model'] == model]
        stats['confusion'][model] = pd.crosstab(
            model_df['author_answer'],
            model_df['llm_answer'],
            normalize='index'
        ).round(3)

    return stats


def plot_overall_accuracy(df):
    """Plot overall accuracy by model."""
    fig, ax = plt.subplots(figsize=(10, 6))

    accuracy_by_model = df.groupby('model')['agreement'].mean().sort_values(ascending=False)

    bars = ax.bar(range(len(accuracy_by_model)), accuracy_by_model.values)
    ax.set_xticks(range(len(accuracy_by_model)))
    ax.set_xticklabels(accuracy_by_model.index, rotation=45, ha='right')
    ax.set_ylabel('Precisión (Tasa de Concordancia)')
    ax.set_xlabel('Modelo')
    ax.set_title('Precisión General de los Modelos en la Evaluación del Checklist')
    ax.set_ylim([0, 1])

    # Add percentage labels on bars
    for bar, val in zip(bars, accuracy_by_model.values):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.02,
                f'{val:.1%}', ha='center', va='bottom')

    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_OUTPUT_DIR, 'overall_accuracy.png'), dpi=300, bbox_inches='tight')
    plt.close()

    print("✓ Saved: overall_accuracy.png")


def plot_accuracy_by_question(df):
    """Plot accuracy by question for each model as a heatmap."""
    fig, ax = plt.subplots(figsize=(14, 10))

    pivot = df.groupby(['question', 'model'])['agreement'].mean().unstack()

    sns.heatmap(pivot, annot=True, fmt='.2%', cmap='RdYlGn',
                center=0.5, vmin=0, vmax=1, ax=ax,
                cbar_kws={'label': 'Tasa de Concordancia'})

    ax.set_title('Tasa de Concordancia por Pregunta y Modelo')
    ax.set_xlabel('Modelo')
    ax.set_ylabel('Pregunta')

    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_OUTPUT_DIR, 'accuracy_by_question_heatmap.png'),
                dpi=300, bbox_inches='tight')
    plt.close()

    print("✓ Saved: accuracy_by_question_heatmap.png")


def plot_question_difficulty(df):
    """Plot questions ranked by difficulty (disagreement rate)."""
    fig, ax = plt.subplots(figsize=(12, 8))

    question_accuracy = df.groupby('question')['agreement'].agg(['mean', 'count'])
    question_accuracy = question_accuracy.sort_values('mean')

    bars = ax.barh(range(len(question_accuracy)), question_accuracy['mean'].values)
    ax.set_yticks(range(len(question_accuracy)))
    ax.set_yticklabels(question_accuracy.index)
    ax.set_xlabel('Tasa de Concordancia')
    ax.set_title('Preguntas Ordenadas por Tasa de Concordancia (Más Fácil a Más Difícil)')
    ax.set_xlim([0, 1])

    for bar, val in zip(bars, question_accuracy['mean'].values):
        if val >= 0.8:
            bar.set_color('green')
        elif val >= 0.5:
            bar.set_color('orange')
        else:
            bar.set_color('red')

        ax.text(val + 0.02, bar.get_y() + bar.get_height() / 2,
                f'{val:.1%}', va='center')

    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_OUTPUT_DIR, 'question_difficulty.png'),
                dpi=300, bbox_inches='tight')
    plt.close()

    print("✓ Saved: question_difficulty.png")


def plot_confusion_matrices(df, models):
    """
    Plot confusion matrices for each model arranged in a grid.
    Up to 3 columns; rows are added automatically.
    """
    n_models = len(models)
    n_cols = min(n_models, 3)
    n_rows = math.ceil(n_models / n_cols)

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(6 * n_cols, 5 * n_rows),
        squeeze=False,          # always return 2-D array of axes
    )

    for idx, model in enumerate(models):
        row, col = divmod(idx, n_cols)
        ax = axes[row][col]

        model_df = df[df['model'] == model]
        confusion = pd.crosstab(
            model_df['author_answer'],
            model_df['llm_answer'],
            normalize='index'
        )

        sns.heatmap(confusion, annot=True, fmt='.2%', cmap='Blues', ax=ax,
                    vmin=0, vmax=1, cbar_kws={'label': 'Proportion'})
        ax.set_title(f'{model}', fontsize=11, fontweight='bold')
        ax.set_xlabel('Respuesta del LLM')
        ax.set_ylabel('Respuesta del Autor')

    # Hide any unused axes in the last row
    for idx in range(n_models, n_rows * n_cols):
        row, col = divmod(idx, n_cols)
        axes[row][col].set_visible(False)

    fig.suptitle('Matrices de Confusión — Respuesta del Autor vs. Respuesta del LLM',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_OUTPUT_DIR, 'confusion_matrices.png'),
                dpi=300, bbox_inches='tight')
    plt.close()

    print("✓ Saved: confusion_matrices.png")


def plot_answer_distribution(df):
    """Plot distribution of answers (Yes/No/NA) by model, with Author as another model."""
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))

    llm_dist = df.groupby(['model', 'llm_answer']).size().unstack(fill_value=0)
    llm_dist_pct = llm_dist.div(llm_dist.sum(axis=1), axis=0) * 100

    author_dist = df.groupby('author_answer').size()
    author_dist_pct = (author_dist / author_dist.sum() * 100).to_frame().T
    author_dist_pct.index = ['Authors']

    all_cols = llm_dist_pct.columns.union(author_dist_pct.columns)
    llm_dist_pct = llm_dist_pct.reindex(columns=all_cols, fill_value=0)
    author_dist_pct = author_dist_pct.reindex(columns=all_cols, fill_value=0)

    combined = pd.concat([llm_dist_pct, author_dist_pct])

    combined.plot(
        kind='bar',
        stacked=True,
        ax=ax,
        color=['#95a5a6', '#e74c3c', '#2ecc71']
    )

    ax.set_title('Distribución de Respuestas')
    ax.set_xlabel('Modelo')
    ax.set_ylabel('Porcentaje')
    ax.legend(title='Respuesta', bbox_to_anchor=(1.05, 1))
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

    plt.tight_layout()
    plt.savefig(
        os.path.join(ANALYSIS_OUTPUT_DIR, 'answer_distribution.png'),
        dpi=300, bbox_inches='tight'
    )
    plt.close()

    print("✓ Saved: answer_distribution.png")


def plot_per_paper_score_distribution(df):
    """
    Plot how many papers each model scored at each level (0–N correct out of
    however many questions were actually compared for that paper).
    """
    paper_scores = (
        df.groupby(['model', 'paper_hash'])['agreement']
        .agg(correct='sum', total='count')
        .reset_index()
    )

    paper_scores['score_label'] = (
        paper_scores['correct'].astype(int).astype(str) + ' / ' +
        paper_scores['total'].astype(int).astype(str)
    )

    score_counts = (
        paper_scores.groupby(['model', 'score_label'])
        .size()
        .reset_index(name='num_papers')
    )

    all_totals = sorted(paper_scores['total'].unique(), reverse=True)

    all_labels = []
    for t in all_totals:
        for c in range(int(t), -1, -1):
            all_labels.append(f'{c} / {int(t)}')

    models = sorted(df['model'].unique())

    full_grid = pd.DataFrame(
        [(m, lbl) for m in models for lbl in all_labels],
        columns=['model', 'score_label']
    )
    score_counts = full_grid.merge(score_counts, on=['model', 'score_label'], how='left').fillna(0)
    score_counts['num_papers'] = score_counts['num_papers'].astype(int)

    pivot = score_counts.pivot(index='score_label', columns='model', values='num_papers')
    pivot = pivot.reindex(all_labels)
    pivot = pivot.loc[pivot.sum(axis=1) > 0]

    n_models = len(models)
    bar_width = 0.8 / n_models
    x = np.arange(len(pivot))
    colours = plt.cm.tab10.colors[:n_models]

    fig, (ax_bars, ax_cum) = plt.subplots(
        2, 1, figsize=(max(14, len(pivot) * 0.7 + 4), 10),
        sharex=True, gridspec_kw={'hspace': 0.08}
    )

    for i, model in enumerate(models):
        offset = (i - n_models / 2 + 0.5) * bar_width
        ax_bars.bar(x + offset, pivot[model].values, width=bar_width,
                    label=model, color=colours[i], edgecolor='white', linewidth=0.6)

    ax_bars.set_ylabel('Número de artículos', fontsize=11)
    ax_bars.set_title('Distribución de Puntuaciones por Artículo — Comparación de Modelos', fontsize=14, pad=14)
    ax_bars.set_ylim(bottom=0)
    ax_bars.tick_params(labelbottom=False)

    for i, model in enumerate(models):
        cumulative = pivot[model].values.cumsum()
        ax_cum.plot(x, cumulative, color=colours[i],
                    linewidth=2.2, marker='o', markersize=4, label=model)

    ax_cum.set_ylabel('Artículos con ≥ esta puntuación\n(acumulado)', fontsize=11)
    ax_cum.set_xlabel('Puntuación (correctas / preguntas totales)', fontsize=11)
    ax_cum.set_xticks(x)
    ax_cum.set_xticklabels(pivot.index, rotation=45, ha='right', fontsize=9)
    ax_cum.set_ylim(bottom=0)

    handles, labels = ax_bars.get_legend_handles_labels()
    fig.legend(handles, labels,
               loc='center left', bbox_to_anchor=(-0.02, 0.5),
               fontsize=9, framealpha=0.95, title='Modelo', title_fontsize=10)

    fig.subplots_adjust(left=0.28)
    plt.savefig(os.path.join(ANALYSIS_OUTPUT_DIR, 'per_paper_score_distribution.png'),
                dpi=300, bbox_inches='tight')
    plt.close()

    print("✓ Saved: per_paper_score_distribution.png")


# ---------------------------------------------------------------------------
# NEW PLOTS
# ---------------------------------------------------------------------------

def plot_per_question_accuracy_with_errorbars(df):
    """
    Bar chart of per-question accuracy averaged across models, with error bars
    showing the standard deviation across models.  This highlights questions
    where models *disagree with each other* (high spread) vs. questions that
    are consistently easy or hard.
    """
    q_model = (
        df.groupby(['question', 'model'])['agreement']
        .mean()
        .unstack()          # shape: questions × models
    )

    means = q_model.mean(axis=1).sort_values(ascending=False)
    stds  = q_model.std(axis=1).reindex(means.index)

    fig, ax = plt.subplots(figsize=(14, 7))

    colors = ['#e74c3c' if m < 0.5 else '#f39c12' if m < 0.8 else '#2ecc71'
              for m in means.values]

    bars = ax.bar(range(len(means)), means.values, yerr=stds.values,
                  color=colors, edgecolor='white', linewidth=0.6,
                  error_kw=dict(ecolor='#333333', lw=1.5, capsize=4, capthick=1.5))

    ax.set_xticks(range(len(means)))
    ax.set_xticklabels(means.index, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Tasa de Concordancia Media (± desv. típica entre modelos)')
    ax.set_ylim(0, 1.12)
    ax.axhline(0.8, color='#27ae60', linestyle='--', linewidth=1, alpha=0.7, label='Umbral 80%')
    ax.axhline(0.5, color='#e74c3c', linestyle='--', linewidth=1, alpha=0.7, label='Umbral 50%')
    ax.set_title('Precisión por Pregunta con Varianza entre Modelos\n'
                 '(barras de error = desviación típica entre modelos)', fontsize=13)
    ax.legend(fontsize=9)

    for i, (val, std) in enumerate(zip(means.values, stds.values)):
        ax.text(i, val + (std if not np.isnan(std) else 0) + 0.03,
                f'{val:.0%}', ha='center', va='bottom', fontsize=7.5)

    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_OUTPUT_DIR, 'per_question_accuracy_errorbars.png'),
                dpi=300, bbox_inches='tight')
    plt.close()

    print("✓ Saved: per_question_accuracy_errorbars.png")


def plot_pairwise_model_agreement(df):
    """
    Pairwise agreement heatmap between models — i.e. how often do two models
    give the *same* answer for the same (paper, question) pair, regardless of
    whether either is correct.  Useful to spot models that behave similarly.
    """
    models = sorted(df['model'].unique())
    n = len(models)

    if n < 2:
        print("⚠  Omitiendo concordancia entre pares: se necesitan al menos 2 modelos.")
        return

    # Pivot to wide format: one column per model, rows = (paper, question)
    wide = df.pivot_table(
        index=['paper_hash', 'question'],
        columns='model',
        values='llm_answer',
        aggfunc='first'
    )

    agreement_matrix = pd.DataFrame(np.nan, index=models, columns=models)

    for i, m1 in enumerate(models):
        for j, m2 in enumerate(models):
            if i == j:
                agreement_matrix.loc[m1, m2] = 1.0
                continue
            if m1 not in wide.columns or m2 not in wide.columns:
                continue
            both = wide[[m1, m2]].dropna()
            if len(both) == 0:
                continue
            agreement_matrix.loc[m1, m2] = (both[m1] == both[m2]).mean()

    fig, ax = plt.subplots(figsize=(max(6, n * 1.5), max(5, n * 1.4)))

    mask = np.eye(n, dtype=bool)   # mask diagonal so it doesn't dominate color scale
    sns.heatmap(
        agreement_matrix.astype(float),
        annot=True, fmt='.2%', cmap='coolwarm',
        vmin=0, vmax=1, ax=ax,
        mask=mask,
        cbar_kws={'label': 'Tasa de Concordancia entre Respuestas'},
        linewidths=0.5
    )
    # Fill diagonal with a neutral colour manually
    for i in range(n):
        ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=True, color='#d5d8dc', lw=0))
        ax.text(i + 0.5, i + 0.5, '—', ha='center', va='center', fontsize=11)

    ax.set_title('Concordancia entre Pares de Modelos\n(frecuencia con la que dos modelos dan la misma respuesta)',
                 fontsize=13)
    ax.set_xlabel('Modelo')
    ax.set_ylabel('Modelo')

    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_OUTPUT_DIR, 'pairwise_model_agreement.png'),
                dpi=300, bbox_inches='tight')
    plt.close()

    print("✓ Saved: pairwise_model_agreement.png")


def plot_answer_bias(df):
    """
    For each model, show the *signed* difference in answer-rate vs. authors
    for each answer category (Yes / No / NA).

    A positive bar means the model over-uses that answer relative to authors;
    negative means it under-uses it.  This makes systematic biases immediately
    visible (e.g. "Model X is too lenient — it says Yes far more than authors").
    """
    answer_cats = sorted(df['author_answer'].unique())   # e.g. ['NA', 'No', 'Yes']
    models = sorted(df['model'].unique())

    # Author baseline proportions
    author_props = df.groupby('author_answer').size() / len(df)
    author_props = author_props.reindex(answer_cats, fill_value=0)

    # LLM proportions per model
    rows = []
    for model in models:
        mdf = df[df['model'] == model]
        llm_props = mdf.groupby('llm_answer').size() / len(mdf)
        llm_props = llm_props.reindex(answer_cats, fill_value=0)
        for cat in answer_cats:
            rows.append({
                'model': model,
                'answer': cat,
                'bias': llm_props[cat] - author_props[cat]
            })

    bias_df = pd.DataFrame(rows)
    pivot = bias_df.pivot(index='answer', columns='model', values='bias')

    n_models = len(models)
    fig, ax = plt.subplots(figsize=(max(8, n_models * 2.5), 5))

    bar_width = 0.7 / n_models
    x = np.arange(len(answer_cats))
    colours = plt.cm.tab10.colors[:n_models]

    for i, model in enumerate(models):
        offset = (i - n_models / 2 + 0.5) * bar_width
        vals = pivot[model].values
        bar_colors = ['#e74c3c' if v > 0 else '#3498db' for v in vals]
        bars = ax.bar(x + offset, vals, width=bar_width, label=model,
                      color=colours[i], edgecolor='white', linewidth=0.5, alpha=0.85)

    ax.axhline(0, color='black', linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(answer_cats, fontsize=11)
    ax.set_ylabel('Δ Proporción vs. Autores')
    ax.set_title('Sesgo de Respuesta Respecto a los Autores\n'
                 '(positivo = el modelo usa esta respuesta más de lo esperado, negativo = menos)',
                 fontsize=13)
    ax.legend(title='Modelo', bbox_to_anchor=(1.01, 1), loc='upper left')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:+.1%}'))

    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_OUTPUT_DIR, 'answer_bias.png'),
                dpi=300, bbox_inches='tight')
    plt.close()

    print("✓ Saved: answer_bias.png")


# ---------------------------------------------------------------------------

def save_statistics_report(stats, models_data):
    """Save a text report with key statistics."""
    report_path = os.path.join(ANALYSIS_OUTPUT_DIR, 'statistics_report.txt')

    with open(report_path, 'w') as f:
        f.write("=" * 60 + "\n")
        f.write("ANÁLISIS DEL CHECKLIST DE NeurIPS - INFORME DE ESTADÍSTICAS\n")
        f.write("=" * 60 + "\n\n")

        f.write("PRECISIÓN GENERAL POR MODELO\n")
        f.write("-" * 60 + "\n")
        f.write(stats['by_model'].to_string())
        f.write("\n\n")

        f.write("PRECISIÓN POR PREGUNTA (Top 10 Mejores)\n")
        f.write("-" * 60 + "\n")
        f.write(stats['by_question'].head(10).to_string())
        f.write("\n\n")

        f.write("PRECISIÓN POR PREGUNTA (Top 10 Peores)\n")
        f.write("-" * 60 + "\n")
        f.write(stats['by_question'].tail(10).to_string())
        f.write("\n\n")

        for model, confusion in stats['confusion'].items():
            f.write(f"MATRIZ DE CONFUSIÓN - {model}\n")
            f.write("-" * 60 + "\n")
            f.write(confusion.to_string())
            f.write("\n\n")

        f.write("INFORMACIÓN DEL CONJUNTO DE DATOS\n")
        f.write("-" * 60 + "\n")
        for model_name, results in models_data.items():
            f.write(f"{model_name}: {len(results)} artículos analizados\n")

    print(f"✓ Saved: statistics_report.txt")


def export_to_csv(df):
    """Export the comparison dataframe to CSV."""
    csv_path = os.path.join(ANALYSIS_OUTPUT_DIR, 'full_comparison.csv')
    df.to_csv(csv_path, index=False)
    print(f"✓ Saved: full_comparison.csv")


def analyze_all_models():
    """Main analysis function."""
    print(f"\n{'=' * 60}")
    print(f"Análisis del Checklist de NeurIPS")
    print(f"{'=' * 60}\n")

    models = get_all_model_directories()

    if not models:
        print("Error: ¡No se encontraron directorios de modelos!")
        return

    print(f"Se encontraron {len(models)} modelo(s): {', '.join(models)}\n")

    print("Cargando resultados...")
    models_data = {}
    for model in models:
        results = load_all_results(model)
        models_data[model] = results
        print(f"  {model}: {len(results)} artículos")

    print()

    print("Creando dataframe de comparación...")
    df = create_comparison_dataframe(models_data)
    print(f"  Comparaciones totales: {len(df):,}")
    print(f"  Artículos únicos: {df['paper_hash'].nunique():,}")
    print()

    print("Calculando estadísticas...")
    stats = calculate_statistics(df)
    print()

    print("Generando visualizaciones...")
    plot_overall_accuracy(df)
    plot_accuracy_by_question(df)
    plot_question_difficulty(df)
    plot_confusion_matrices(df, models)           # ← ahora en cuadrícula
    plot_answer_distribution(df)
    plot_per_paper_score_distribution(df)
    plot_per_question_accuracy_with_errorbars(df) # ← nuevo
    plot_pairwise_model_agreement(df)             # ← nuevo
    plot_answer_bias(df)                          # ← nuevo
    print()

    print("Guardando informes...")
    save_statistics_report(stats, models_data)
    export_to_csv(df)
    print()

    print("=" * 60)
    print("RESUMEN")
    print("=" * 60)
    print(f"Modelos analizados: {len(models)}")
    print(f"Artículos totales: {df['paper_hash'].nunique()}")
    print(f"Comparaciones totales: {len(df):,}")
    print(f"\nModelo con mejor rendimiento:")
    best_model = stats['by_model']['accuracy'].idxmax()
    best_accuracy = stats['by_model']['accuracy'].max()
    print(f"  {best_model}: {best_accuracy:.1%} de precisión")
    print(f"\nPregunta más fácil (mayor concordancia):")
    easiest = stats['by_question']['accuracy'].idxmax()
    easiest_acc = stats['by_question']['accuracy'].max()
    print(f"  {easiest}: {easiest_acc:.1%}")
    print(f"\nPregunta más difícil (menor concordancia):")
    hardest = stats['by_question']['accuracy'].idxmin()
    hardest_acc = stats['by_question']['accuracy'].min()
    print(f"  {hardest}: {hardest_acc:.1%}")
    print(f"\nTodos los resultados guardados en: {ANALYSIS_OUTPUT_DIR}/")
    print("=" * 60 + "\n")

    return df, stats


# Run the analysis
if __name__ == "__main__":
    df, stats = analyze_all_models()


Análisis del Checklist de NeurIPS

Se encontraron 9 modelo(s): gemma3:27b, gpt-oss:20b, deepseek-r1:8b, mistral-nemo:12b, gemma3:12b, gemma3:4b, hermes3:8b, qwen3.5:9b, llama3.1:8b

Cargando resultados...
  gemma3:27b: 3747 artículos
  gpt-oss:20b: 3746 artículos
  deepseek-r1:8b: 3748 artículos
  mistral-nemo:12b: 3748 artículos
  gemma3:12b: 3747 artículos
  gemma3:4b: 3747 artículos
  hermes3:8b: 3748 artículos
  qwen3.5:9b: 3747 artículos
  llama3.1:8b: 3747 artículos

Creando dataframe de comparación...
  Comparaciones totales: 505,875
  Artículos únicos: 3,748

Calculando estadísticas...

Generando visualizaciones...
✓ Saved: overall_accuracy.png
✓ Saved: accuracy_by_question_heatmap.png
✓ Saved: question_difficulty.png
✓ Saved: confusion_matrices.png
✓ Saved: answer_distribution.png
✓ Saved: per_paper_score_distribution.png
✓ Saved: per_question_accuracy_errorbars.png
✓ Saved: pairwise_model_agreement.png
✓ Saved: answer_bias.png

Guardando informes...
✓ Saved: statistics_repor